# CS 5330 — Final In-Class Activity: Transfer Learning on Produce Images

**Week 3 · Pattern Recognition & Computer Vision**

### What you'll do today

You'll take a CNN that has already been trained on 1.3 million ImageNet photos, and repurpose it — with only a few hundred images and a few minutes of training — to recognize a handful of produce types. This is **transfer learning**: instead of learning everything from scratch, you reuse the general-purpose visual features (edges, textures, shapes) a big pretrained network already learned, and only retrain the part that's specific to your task.

### Why this matters for your own project

Back in Week 2, you started photographing your own produce dataset — and you may have noticed it's small: maybe a few dozen images per item, not thousands. That's normal, and it's exactly the situation transfer learning is built for. Today's activity uses a larger public produce dataset so we have enough images to see clear results in one class session, but everything you learn here — the code, the decisions, the trade-offs — applies directly when you point this same pipeline at your own, smaller dataset later.

### How this notebook is organized

- Regular sections explain a concept and give you working code.
- Sections marked **🎛️ Tweak this** contain a parameter *you* should change, then re-run the cells below it to see what happens.
- Sections marked **✍️ Your answer** are where you record what you observed. There's no single correct answer — this activity is graded on effort and genuine engagement, not on hitting a specific accuracy number.

### Before you start

Go to **Runtime → Change runtime type** and set Hardware accelerator to **GPU** (T4 is fine). Training will work on CPU too, just slower.

## Step 0 — Imports and device check

Colab already has PyTorch and torchvision installed, so there's nothing to `pip install` here. We just import what we need and check whether a GPU is available. `device` gets reused everywhere below — every tensor and model we create has to live on the same device, or PyTorch will raise an error.

In [ ]:
import os
import copy
import time
import shutil

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, models, transforms

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
if device.type == "cpu":
    print("No GPU detected — this will still work, just more slowly. "
          "Double check Runtime > Change runtime type > GPU.")

## Step 1 — Get the data

We're using a public produce-image dataset (originally built for fruit recognition benchmarks) that ships as plain folders of images, one folder per class — a format called **`ImageFolder`** layout, which is what torchvision's `datasets.ImageFolder` expects out of the box. Each image shows a single piece of produce against a plain background, which keeps today's activity focused on the transfer-learning workflow rather than on messy real-world photos (your own Week 2 dataset is messier — and that's a great thing to compare against later).

We pull it with a shallow `git clone` (`--depth 1` skips the repo's full commit history, which we don't need and which would make the download much bigger).

> **Heads up:** this step needs network access, which only exists once this notebook is actually running in Colab — it isn't something that could be pre-tested elsewhere. If the folder names printed below don't match what you expect, run `!ls fruits_data` yourself and adjust `train_dir` / `test_dir` accordingly.

> **This will take a few minutes.** Even with `--depth 1`, this repository is around 2.2 GB — it contains all 131 classes, not just the 6 we'll use. Git can download just the folders we need (a *sparse checkout*), but that takes several extra setup commands, so for simplicity we clone everything and pick our classes in the cells below. **Kick this cell off first, before anything else** (e.g., while going over the instructions above as a class), so the download is done by the time everyone's ready to run code.

In [ ]:
!git clone --depth 1 https://github.com/Horea94/Fruit-Images-Dataset.git fruits_data

In [ ]:
DATA_ROOT = "fruits_data"
train_dir = os.path.join(DATA_ROOT, "Training")
test_dir = os.path.join(DATA_ROOT, "Test")

print("Training dir found:", os.path.isdir(train_dir))
print("Test dir found:", os.path.isdir(test_dir))

if not (os.path.isdir(train_dir) and os.path.isdir(test_dir)):
    print("\nExpected folder names weren't found. Here's what actually got cloned:")
    print(os.listdir(DATA_ROOT))
    print("\nUpdate train_dir / test_dir above to match, then re-run this cell.")

### 🎛️ Tweak this: which classes to use

This dataset has a *lot* of classes — many of them are near-duplicates of each other (different varieties of the same fruit, e.g. several kinds of apple). Run the cell below to see what's actually available.

We're picking classes deliberately here, not arbitrarily: **apple, blueberry, grape, pear, plum, and potato** — because those are exactly six of the produce types your class photographed for Step 7 later. That overlap is what makes the later "test on your own photos" comparison meaningful: same produce types, wildly different image sources.

Four of your class's photographed types — **blackberry, cranberry, pumpkin, squash** — aren't reliably present in this public dataset, so the model won't be trained to recognize them at all. That's worth sitting with rather than working around: a classifier can only ever predict classes it was trained on. If you feed it a pumpkin photo, it will *still* confidently output one of its six known classes — just a wrong one. (This is a legitimate thing to try and discuss in Step 7 if you're curious, not something to avoid.)

In [ ]:
all_classes = sorted(os.listdir(train_dir))
print(f"Found {len(all_classes)} classes in the training set.")
print("First 25, alphabetically:")
for name in all_classes[:25]:
    print(" -", name)

In [ ]:
def find_class(prefix, classes):
    """Return the first class name starting with `prefix` (case-insensitive), or None."""
    matches = sorted(c for c in classes if c.lower().startswith(prefix.lower()))
    return matches[0] if matches else None


# 🎛️ TWEAK ME: change or add to this list to select different produce types.
# We search by prefix rather than hardcoding exact variety names, since this
# dataset's precise folder names (e.g. "Apple Golden 1" vs "Apple Braeburn")
# can shift between versions of the dataset.
DESIRED_FRUITS = ["Apple", "Blueberry", "Grape", "Pear", "Plum", "Potato"]

SELECTED_CLASSES = []
for fruit in DESIRED_FRUITS:
    match = find_class(fruit, all_classes)
    if match:
        SELECTED_CLASSES.append(match)
        print(f"{fruit:12s} -> using '{match}'")
    else:
        print(f"{fruit:12s} -> no match found in this dataset! Check `all_classes` above.")

print("\nFinal SELECTED_CLASSES:", SELECTED_CLASSES)

### Building a subset

`ImageFolder` treats *every* subfolder it finds as a class, so to train on only our chosen classes we build a small subset directory containing just those folders. We use symbolic links (`os.symlink`) rather than copying files — same end result for training, but instant and using no extra disk space.

In [ ]:
def build_subset(src_dir, dst_dir, classes):
    if os.path.exists(dst_dir):
        shutil.rmtree(dst_dir)
    os.makedirs(dst_dir, exist_ok=True)
    for class_name in classes:
        src = os.path.abspath(os.path.join(src_dir, class_name))
        dst = os.path.join(dst_dir, class_name)
        os.symlink(src, dst)

build_subset(train_dir, "subset/Training", SELECTED_CLASSES)
build_subset(test_dir, "subset/Test", SELECTED_CLASSES)

print("Subset ready:")
for c in SELECTED_CLASSES:
    n_train = len(os.listdir(os.path.join("subset/Training", c)))
    n_test = len(os.listdir(os.path.join("subset/Test", c)))
    print(f"  {c}: {n_train} training images, {n_test} test images")

## Step 2 — Preprocessing and augmentation

Every image needs to be turned into a tensor of the right shape and value range before it can go into the network. Two things matter here specifically because we're using a **pretrained** backbone:

1. **Resizing to 224×224.** The pretrained ResNet was trained on images of this size. Feed it something a different size and the math inside still works (conv layers don't care about input size), but the very last spatial dimensions going into the final layer will be wrong.
2. **Normalizing with ImageNet's mean and standard deviation**, not our own dataset's. The pretrained filters learned to expect pixel values centered around ImageNet's statistics — feed them differently-scaled input and every downstream computation is subtly off, even though nothing crashes.

For the *training* set only, we also add light **data augmentation** — small random rotations and flips. This artificially creates more visual variety from the same images, which matters most for small datasets (like the one you'll eventually build yourself), where the model can otherwise memorize exact training images instead of learning general features.

In [ ]:
# 🎛️ TWEAK ME: augmentation strength
ROTATION_DEGREES = 15   # try 0 (no rotation) or 45 (aggressive) and compare later
HFLIP_PROB = 0.5         # probability of a horizontal flip; set to 0.0 to disable

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomRotation(ROTATION_DEGREES),
    transforms.RandomHorizontalFlip(p=HFLIP_PROB),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

# No randomness at evaluation time — we want a consistent, repeatable measurement.
eval_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

### Loading the data

In [ ]:
# 🎛️ TWEAK ME
BATCH_SIZE = 32

train_dataset = datasets.ImageFolder("subset/Training", transform=train_transforms)
test_dataset = datasets.ImageFolder("subset/Test", transform=eval_transforms)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

class_names = train_dataset.classes
num_classes = len(class_names)

print(f"{num_classes} classes: {class_names}")
print(f"{len(train_dataset)} training images, {len(test_dataset)} test images")

### A quick look at what the model will actually see

Worth pausing on: the images below have already been resized, augmented, and normalized. We reverse the normalization just so they display with sensible colors — the network never sees this "undone" version, only the normalized tensor.

In [ ]:
def denormalize(tensor):
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std = torch.tensor(IMAGENET_STD).view(3, 1, 1)
    return tensor * std + mean

images, labels = next(iter(train_loader))

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for i, ax in enumerate(axes.flat):
    if i >= len(images):
        break
    img = denormalize(images[i]).permute(1, 2, 0).numpy().clip(0, 1)
    ax.imshow(img)
    ax.set_title(class_names[labels[i]], fontsize=10)
    ax.axis("off")
plt.suptitle("A batch of augmented training images")
plt.tight_layout()
plt.show()

## Step 3 — Load the pretrained backbone

We're using **ResNet18** — the same family of architecture from the Week 3 lecture. `weights=ResNet18_Weights.DEFAULT` downloads a version of ResNet18 already trained on the full 1000-class ImageNet dataset. Its convolutional layers have already learned a hierarchy of visual features: edges and textures in the earliest layers, increasingly complex shapes and parts deeper in.

That original network ends in a fully-connected layer producing 1000 outputs — one per ImageNet class. We don't want 1000 classes, we want `num_classes` (however many you selected above), so we **replace only that final layer** with a fresh one sized for our task. Everything before it — all the feature-extracting convolutional layers — we keep exactly as they are.

In [ ]:
try:
    weights = models.ResNet18_Weights.DEFAULT
    model = models.resnet18(weights=weights)
except AttributeError:
    # Fallback for older torchvision versions.
    model = models.resnet18(pretrained=True)

print("Original final layer (built for 1000 ImageNet classes):")
print(model.fc)

model.fc = nn.Linear(model.fc.in_features, num_classes)

print("\nReplaced final layer (built for our task):")
print(model.fc)

model = model.to(device)

## Step 4 — Freeze the backbone, or fine-tune it?

There are two common strategies for reusing a pretrained network, and they're different enough to be worth naming precisely:

- **Feature extraction (freeze the backbone).** Every pretrained layer keeps its exact weights, frozen — only the new final layer learns anything. This is fast, works well with very little data, and is hard to mess up.
- **Fine-tuning.** The pretrained weights are allowed to update too, usually with a small learning rate so we don't wreck the useful features they already encode. This can reach higher accuracy, especially if your images look quite different from ImageNet's, but it needs more data and more care.

In PyTorch, whether a parameter updates during training is controlled by its `requires_grad` flag. Setting it to `False` tells autograd not to bother computing gradients for that parameter — no gradient, no update.

In [ ]:
# 🎛️ TWEAK ME: feature extraction vs. fine-tuning
FREEZE_BACKBONE = True   # True = only the new final layer trains; False = fine-tune everything

for name, param in model.named_parameters():
    if "fc" in name:
        param.requires_grad = True   # the new final layer always trains
    else:
        param.requires_grad = not FREEZE_BACKBONE

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable:,} / {total:,} ({100 * trainable / total:.1f}%)")

## Step 5 — Loss, optimizer, and the training loop

For multi-class classification, **cross-entropy loss** is the standard choice — it compares the network's predicted probability distribution (after an internal softmax, exactly like the one from lecture) against the true class, and penalizes confident wrong answers more heavily than uncertain ones.

The **learning rate** controls how big a step the optimizer takes on each update. This interacts directly with the freeze/fine-tune choice above: when fine-tuning pretrained weights, a learning rate that's too large can undo the useful patterns those weights already encode, sometimes within a single epoch. When only training a fresh final layer, a slightly larger learning rate is usually fine since there's nothing pretrained to damage.

In [ ]:
# 🎛️ TWEAK ME
LEARNING_RATE = 0.001   # try 0.01 or 0.0001 later and compare the loss curves
NUM_EPOCHS = 8           # more epochs = more training time, with diminishing returns

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LEARNING_RATE,
)

### The training loop

This function alternates between a **training phase** (gradients on, weights update) and a **validation phase** (gradients off, weights frozen, just measuring performance) once per epoch. We keep a copy of whichever epoch's weights scored the best validation accuracy, since the *last* epoch isn't always the *best* one — a model can start overfitting (memorizing the training set at the expense of generalizing) partway through training.

In [ ]:
def train_model(model, train_loader, test_loader, criterion, optimizer, num_epochs):
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    best_acc = 0.0
    best_weights = copy.deepcopy(model.state_dict())

    for epoch in range(num_epochs):
        start = time.time()

        for phase, loader in [("train", train_loader), ("val", test_loader)]:
            model.train() if phase == "train" else model.eval()

            running_loss = 0.0
            running_correct = 0

            for inputs, labels in loader:
                inputs, labels = inputs.to(device), labels.to(device)
                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == "train"):
                    outputs = model(inputs)
                    loss = criterion(outputs, labels)
                    _, preds = torch.max(outputs, 1)

                    if phase == "train":
                        loss.backward()
                        optimizer.step()

                running_loss += loss.item() * inputs.size(0)
                running_correct += torch.sum(preds == labels.data).item()

            epoch_loss = running_loss / len(loader.dataset)
            epoch_acc = running_correct / len(loader.dataset)
            history[f"{phase}_loss"].append(epoch_loss)
            history[f"{phase}_acc"].append(epoch_acc)

            if phase == "val" and epoch_acc > best_acc:
                best_acc = epoch_acc
                best_weights = copy.deepcopy(model.state_dict())

        elapsed = time.time() - start
        print(
            f"Epoch {epoch + 1}/{num_epochs} ({elapsed:.1f}s) — "
            f"train loss {history['train_loss'][-1]:.3f} acc {history['train_acc'][-1]:.3f} | "
            f"val loss {history['val_loss'][-1]:.3f} acc {history['val_acc'][-1]:.3f}"
        )

    print(f"\nBest validation accuracy this run: {best_acc:.3f}")
    model.load_state_dict(best_weights)
    return model, history

In [ ]:
model, history = train_model(model, train_loader, test_loader, criterion, optimizer, NUM_EPOCHS)

## Step 6 — Look at the results

### Loss and accuracy curves

What to look for: if training accuracy keeps climbing while validation accuracy flattens or drops, that's the signature of **overfitting** — the model is memorizing training images rather than learning generalizable features.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history["train_loss"], label="train")
axes[0].plot(history["val_loss"], label="val")
axes[0].set_title("Loss")
axes[0].set_xlabel("Epoch")
axes[0].legend()

axes[1].plot(history["train_acc"], label="train")
axes[1].plot(history["val_acc"], label="val")
axes[1].set_title("Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].legend()

plt.tight_layout()
plt.show()

### Confusion matrix

Overall accuracy hides *which* classes get confused for each other. A confusion matrix shows every true-class-vs-predicted-class combination — the diagonal is correct predictions, anything off the diagonal is a specific kind of mistake worth looking at.

In [ ]:
model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.to(device)
        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

cm = confusion_matrix(all_labels, all_preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)

fig, ax = plt.subplots(figsize=(6, 6))
disp.plot(ax=ax, xticks_rotation=45, colorbar=False)
plt.title("Confusion matrix (test set)")
plt.tight_layout()
plt.show()

### A few individual predictions

In [ ]:
images, labels = next(iter(test_loader))
inputs = images.to(device)

with torch.no_grad():
    outputs = model(inputs)
    _, preds = torch.max(outputs, 1)

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for i, ax in enumerate(axes.flat):
    if i >= len(images):
        break
    img = denormalize(images[i]).permute(1, 2, 0).numpy().clip(0, 1)
    ax.imshow(img)
    true_label = class_names[labels[i]]
    pred_label = class_names[preds[i]]
    color = "green" if true_label == pred_label else "red"
    ax.set_title(f"true: {true_label}\npred: {pred_label}", fontsize=9, color=color)
    ax.axis("off")
plt.suptitle("Sample predictions (green = correct, red = wrong)")
plt.tight_layout()
plt.show()

## Step 7 — The real test: your own produce photos

Everything so far was trained *and* tested on the same clean, studio-style dataset — plain backgrounds, consistent lighting, one item centered per image. Your own Week 2 photos look nothing like that: different phones, different lighting, cluttered backgrounds, produce photographed at odd angles.

That mismatch between what a model was trained on and what it's actually asked to handle later is called **domain shift**, and it's one of the most common ways real computer vision systems underperform in practice. Testing on your own photos — images the model has never seen, from a visibly different source — is a much more honest measure of whether it actually learned generalizable features, versus just fitting the specific look of the public dataset.

> This section pulls from the class's own produce-photo repo (`LinoNU/cs5330-produce-photos`), organized as one subfolder per produce type inside a nested dataset folder. You'll map those folder names to the trained classes just below.

In [ ]:
# The class's own produce-photo repo
OWN_PHOTOS_REPO_URL = "https://github.com/LinoNU/cs5330-produce-photos.git"

print(f"Cloning {OWN_PHOTOS_REPO_URL} ...")
!git clone --depth 1 {OWN_PHOTOS_REPO_URL} own_photos

In [ ]:
# The repo stores images inside this nested folder rather than at the repo root.
OWN_PHOTOS_DIR = "own_photos/CS 5330 Fall 2026 Class Dataset"   # 🎛️ update if this moves

print("Contents of the dataset folder (these should be your produce-type subfolders):")
print(os.listdir(OWN_PHOTOS_DIR))
print("\nThe model was trained on these classes:")
print(class_names)

### Mapping your folder names to the trained classes

Your own folders (`apple`, `blackberry`, `blueberry`, `cranberry`, `grape`, `pear`, `plum`, `potato`, `pumpkin`, `squash`) are named generically, while the trained model's classes are specific varieties (e.g. `Apple Golden 1` rather than plain `apple`). Rather than relying on folder names matching automatically — fragile, and it fails silently in confusing ways — we make the mapping explicit and resolve it the same way we picked the training classes: by prefix, against whatever `class_names` the model actually ended up with.

Only six of your ten folders have a corresponding trained class. `blackberry`, `cranberry`, `pumpkin`, and `squash` are intentionally left out — the model was never trained to recognize them.

In [ ]:
# 🎛️ TWEAK ME: left side = your folder name, right side = the produce type
# prefix to search for among the trained classes. Add a line here if your
# class photographed additional overlapping produce types.
OWN_FOLDER_TO_FRUIT_PREFIX = {
    "apple": "Apple",
    "blueberry": "Blueberry",
    "grape": "Grape",
    "pear": "Pear",
    "plum": "Plum",
    "potato": "Potato",
    # Not mapped -- no matching trained class in this public dataset:
    # "blackberry", "cranberry", "pumpkin", "squash"
}

OWN_TO_MODEL_CLASS = {}
for own_folder, prefix in OWN_FOLDER_TO_FRUIT_PREFIX.items():
    match = find_class(prefix, class_names)
    if match:
        OWN_TO_MODEL_CLASS[own_folder] = match
    else:
        print(f"No trained class starts with '{prefix}' -- skipping '{own_folder}'.")

print("Final mapping (your folder -> trained class):")
for k, v in OWN_TO_MODEL_CLASS.items():
    print(f"  {k:12s} -> {v}")

### A dataset that respects the model's original class indices

A plain `ImageFolder` on your own photos would assign class indices 0, 1, 2, ... based on *whichever folders happen to be present* — which almost certainly won't line up with the indices the model was actually trained on, especially if you only photographed some of the trained classes. The small dataset class below sidesteps that: it looks up each image's label directly from `class_names` (the master list the model uses), so predictions stay meaningfully comparable no matter which subset of classes your own photos cover.

In [ ]:
class MappedFolderDataset(torch.utils.data.Dataset):
    """
    Loads images from `root/<own_folder_name>/*`, labeling each one using
    `model_classes.index(mapped_class_name)` — so labels line up with a
    model trained on `model_classes`, even if this dataset only covers a
    subset of those classes, and even if the folder names don't match at all.
    """
    def __init__(self, root, own_to_model_class, model_classes, transform=None):
        self.transform = transform
        self.samples = []

        for own_folder, model_class in own_to_model_class.items():
            if model_class not in model_classes:
                print(f"Skipping '{own_folder}': '{model_class}' isn't one of the trained classes.")
                continue

            class_dir = os.path.join(root, own_folder)
            if not os.path.isdir(class_dir):
                print(f"Skipping '{own_folder}': folder not found in {root}.")
                continue

            label = model_classes.index(model_class)
            for fname in os.listdir(class_dir):
                fpath = os.path.join(class_dir, fname)
                if os.path.isfile(fpath):
                    self.samples.append((fpath, label))

        if not self.samples:
            print("Warning: no images matched. Check OWN_TO_MODEL_CLASS and the folder names above.")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        image = Image.open(path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, label


own_dataset = MappedFolderDataset(OWN_PHOTOS_DIR, OWN_TO_MODEL_CLASS, class_names, transform=eval_transforms)
own_loader = DataLoader(own_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

covered = sorted({class_names[label] for _, label in own_dataset.samples})
print(f"Testing on {len(own_dataset)} of your own photos, covering: {covered}")

### Evaluate — and compare against the public test set

In [ ]:
model.eval()
own_preds, own_labels = [], []

with torch.no_grad():
    for inputs, labels in own_loader:
        inputs = inputs.to(device)
        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)
        own_preds.extend(preds.cpu().numpy())
        own_labels.extend(labels.numpy())

own_accuracy = np.mean(np.array(own_preds) == np.array(own_labels))
# Use the BEST epoch's accuracy, not the last one -- train_model() already
# loaded the best epoch's weights back into `model` before returning, so
# comparing against history["val_acc"][-1] could silently compare against a
# different (later, possibly worse) epoch than the one actually being evaluated.
public_accuracy = max(history["val_acc"])

print(f"Public test-set accuracy:   {public_accuracy:.3f}  (best epoch)")
print(f"Your own photos accuracy:   {own_accuracy:.3f}")
print(f"Gap:                        {public_accuracy - own_accuracy:+.3f}")

cm_own = confusion_matrix(own_labels, own_preds, labels=list(range(num_classes)))
disp = ConfusionMatrixDisplay(confusion_matrix=cm_own, display_labels=class_names)

fig, ax = plt.subplots(figsize=(6, 6))
disp.plot(ax=ax, xticks_rotation=45, colorbar=False)
plt.title("Confusion matrix — your own photos")
plt.tight_layout()
plt.show()

## Now go back and compare configurations

The whole point of today is comparing *runs*, not getting one number. Pick at least two of the tweaks below, change the relevant cell, then re-run everything from that cell down through Step 7 (so both the public test accuracy and your own-photos accuracy get updated together). Log both numbers per run in the table below (double-click this cell to edit it).

| Run | Frozen backbone? | Learning rate | Rotation degrees | Public test accuracy | Own-photos accuracy |
|---|---|---|---|---|---|
| 1 (baseline) |  |  |  |  |  |
| 2 |  |  |  |  |  |
| 3 |  |  |  |  |  |
| 4 |  |  |  |  |  |

## Reflection

Answer these based on what you actually observed today — not what you'd expect in theory. If your results don't match the "textbook" expectation, say so and take a guess at why; that's a completely legitimate and useful answer.

**1. Frozen vs. fine-tuned.** Compare `FREEZE_BACKBONE = True` against `FREEZE_BACKBONE = False`. Which reached higher validation accuracy in your runs? Did one train noticeably faster? What would you guess is happening differently inside the network in each case?

*Your answer:*


**2. Learning rate.** Try a learning rate 10× larger than the default. What happened to the loss curve — did it converge faster, get noisier, or fail to come down at all? Did the answer depend on whether the backbone was frozen?

*Your answer:*


**3. Augmentation.** Compare `ROTATION_DEGREES = 0` (no augmentation) against a stronger setting. Did validation accuracy change? Did you see a bigger gap between training and validation accuracy in one case than the other — and if so, what does that gap usually indicate?

*Your answer:*


**4. Number of classes.** If you increase `SELECTED_CLASSES` to include more, visually similar produce types, what happens to overall accuracy? Look at the confusion matrix — is the model confusing specific classes with each other, or making random mistakes evenly across all classes?

*Your answer:*


**5. The domain-shift gap.** Compare the public test-set accuracy against your own-photos accuracy from Step 7. Is there a gap, and if so, how big? Look at the confusion matrix for your own photos specifically — are the mistakes spread evenly, or concentrated on particular classes? What do you think is different about your photos (background, lighting, angle, resolution) that could explain what you're seeing? If you were going to improve accuracy on your own photos specifically, would you reach for more augmentation, fine-tuning instead of freezing, more of your own training images, or something else — and why?

*Your answer:*



## Wrap-up

Today you built a full transfer-learning pipeline: loaded a network pretrained on ImageNet, decided how much of it to retrain, adapted its final layer to a new task, and trained and evaluated it on produce images — all in far less time and data than training a CNN from scratch would need.

A few things worth carrying forward:

- The convolutional layers you didn't touch encode general-purpose visual features (edges, textures, simple shapes) that transfer across almost any image task, not just ImageNet's original 1000 categories.
- Freezing vs. fine-tuning, learning rate, and augmentation strength aren't independent choices — they interact, and the right combination depends on how much data you have and how different your images are from the pretrained network's original training data.
- This exact pipeline — pretrained backbone, replaced final layer, careful choice of what to freeze — is the foundation for the object detection and segmentation architectures coming up later in the course. They add complexity on top of this same core idea.